# Initialization

In [49]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import joblib
import optuna
import json

In [17]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
current_dir = os.getcwd()
current_dir

'e:\\University\\OOP\\FinanceTracking'

In [6]:
if current_dir.split("\\")[-1] != 'notebooks':
    target_path = os.path.join('src', 'ml-service', 'notebooks')
    os.chdir(target_path)

In [7]:
load_dotenv()

True

In [51]:
DB_URL = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"

TEST_SIZE = 0.2 # 20%
RANDOM_STATE = 0

EXPORT_PATH = "../app/models/exports"
MODEL_EXPORT_PATH = EXPORT_PATH + "/model.joblib"
CATEGORIES_EXPORT_PATH = EXPORT_PATH + "/categories.json"

# Data loading

Here you can configure your own data loading pipeline. We'll use our own because our data is stored inside PostgreSQL DB.

This whole notebook training flow will work with any DataFrame that has columns `name` and `category`.

In [9]:
engine = create_engine(DB_URL)

In [10]:
df = pd.read_sql("SELECT * FROM products", engine)

In [11]:
df.head()

,id,name,category,preprocessed,created_at
0,202,"Нектар вишневий Наш Сік 1,93л",Напої,nektar vyshnevyj nash sik,2025-07-12 16:38:21.971804
1,6730,Цукерки Amos 4D Pineapple Burst у формі ананас...,Солодощі,tsukerky amos pineapple burst formi ananasa,2025-07-12 16:38:21.971804
2,6732,Набір подарунковий чашка 380мл з шоколадною ку...,Солодощі,nabir podarunkovyj chashka shokoladnoju kul'koju,2025-07-12 16:38:21.971804
3,6733,Цукерки Toffifee білий шоколад 125г,Солодощі,tsukerky toffifee bilyj shokolad,2025-07-12 16:38:21.971804
4,6734,Соломка Sergini Столична солодка 40г,Солодощі,solomka sergini stolychna solodka,2025-07-12 16:38:21.971804


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127853 entries, 0 to 127852
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   id            127853 non-null  int64         
 1   name          127853 non-null  object        
 2   category      127853 non-null  object        
 3   preprocessed  127853 non-null  object        
 4   created_at    127853 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 4.9+ MB


In [13]:
df = df[["name", "category"]]
df.head()

,name,category
0,"Нектар вишневий Наш Сік 1,93л",Напої
1,Цукерки Amos 4D Pineapple Burst у формі ананас...,Солодощі
2,Набір подарунковий чашка 380мл з шоколадною ку...,Солодощі
3,Цукерки Toffifee білий шоколад 125г,Солодощі
4,Соломка Sergini Столична солодка 40г,Солодощі


In [24]:
df.groupby("category")[["name"]].count().sort_values(by="name", ascending=False)

,name
category,
Алкоголь,13558
Товари для дому,11376
Солодощі,10479
Гігієна та догляд,10085
Одяг та взуття,6677
Товари для дітей,6553
Напої,6462
Молочні продукти,5878
М'ясо та ковбасні вироби,5770


As we can see, dataset is very imbalanced. We'll leave only the most interesting categories and group some similar categories together.

In [27]:
categories_to_group = {
    "Побутова хімія": "Гігієна та догляд",
    "Пекарня": "Кулінарія",
    "Чипси та снеки": "Кулінарія",
    "Консерви": "Кулінарія",
    "Напівфабрикати": "Кулінарія",
    "Риба та морепродукти" : "М'ясо та ковбасні вироби",
    "Жуйки" : "Солодощі"
}

df['category'] = df['category'].replace(categories_to_group)

In [28]:
categories_to_rename = {
    "Кулінарія": "Готова їжа, снеки та напівфабрикати",
    "М'ясо та ковбасні вироби": "М'ясо та риба"
}

df['category'] = df['category'].replace(categories_to_rename)

In [31]:
categories_to_leave = [
    "Алкоголь",
    "Солодощі",
    "Гігієна та догляд",
    "Одяг та взуття",
    "Напої",
    "Молочні продукти",
    "М'ясо та риба",
    "Фрукти та овочі",
    "Бакалія",
    "Готова їжа, снеки та напівфабрикати",
    "Соуси та спеції",
    "Книги",
    "Канцелярія",
    "Тютюнові вироби",
    "Електроніка",
]

df = df[df['category'].isin(categories_to_leave)].copy()

# Preprocessing

In [32]:
import re
import string
import transliterate

punctuation_to_space = string.punctuation.replace("'", "")
space_translation = str.maketrans(dict.fromkeys(punctuation_to_space, " "))
delete_translation = str.maketrans(dict.fromkeys("’«»", ""))

def normalize_ukrainian_text(text: str) -> str:
    """Processes ukrainian text string into a text string that is ready to be used inside a prediction model."""
    text = text.translate(space_translation).translate(delete_translation)
    
    res = transliterate.translit(text, language_code='uk', reversed=True)
    
    # Add spaces before capital letters (for camelCase words)
    res = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', res).lower()

    words = res.split()
    processed_words = []
    
    for word in words:
        if len(word) < 2:
            continue
            
        if any(char.isdigit() for char in word):
            continue
            
        if not re.fullmatch(r"[a-zA-Z'-]{3,40}", word):
            continue
            
        processed_words.append(word)
    
    return ' '.join(processed_words)

In [33]:
df["preprocessed"] = df["name"].apply(normalize_ukrainian_text)
df.head()

,name,category,preprocessed
0,"Нектар вишневий Наш Сік 1,93л",Напої,nektar vyshnevyj nash sik
1,Цукерки Amos 4D Pineapple Burst у формі ананас...,Солодощі,tsukerky amos pineapple burst formi ananasa
2,Набір подарунковий чашка 380мл з шоколадною ку...,Солодощі,nabir podarunkovyj chashka shokoladnoju kul'koju
3,Цукерки Toffifee білий шоколад 125г,Солодощі,tsukerky toffifee bilyj shokolad
4,Соломка Sergini Столична солодка 40г,Солодощі,solomka sergini stolychna solodka


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98143 entries, 0 to 127754
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          98143 non-null  object
 1   category      98143 non-null  object
 2   preprocessed  98143 non-null  object
dtypes: object(3)
memory usage: 3.0+ MB


In [37]:
df.groupby("category")[["name"]].count().sort_values(by="name", ascending=False)

,name
category,
Алкоголь,13558
"Готова їжа, снеки та напівфабрикати",13296
Гігієна та догляд,12802
Солодощі,10530
М'ясо та риба,8490
Одяг та взуття,6677
Напої,6462
Молочні продукти,5878
Фрукти та овочі,4747


# Train/test split

In [35]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    df['preprocessed'], 
    df['category'], 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE, 
    stratify=df['category']
)

# Model training

## Hyperparameter tuning

In [ ]:
def objective(trial: optuna.Trial):
    ngram_max = trial.suggest_int("ngram_max", 2, 5)
    alpha = trial.suggest_float("alpha", 1e-10, 1.0, log=True)
    
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(1, ngram_max))),
        ('classifier', MultinomialNB(alpha=alpha))
    ])
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    # Using f1_macro because dataset is very imbalanced
    score = cross_val_score(
        pipeline, X_train_full, y_train_full, 
        cv=skf, n_jobs=-1, scoring='f1_macro'
    )
    return score.mean()

In [39]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2026-03-08 16:30:22,916] A new study created in memory with name: no-name-a3b99bd1-0078-43e8-95ef-91def890f876


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-03-08 16:30:30,274] Trial 0 finished with value: 0.6519057374932342 and parameters: {'ngram_max': 2, 'alpha': 0.16929208554844727}. Best is trial 0 with value: 0.6519057374932342.
[I 2026-03-08 16:30:42,804] Trial 1 finished with value: 0.9252642837625905 and parameters: {'ngram_max': 5, 'alpha': 0.01839603650780102}. Best is trial 1 with value: 0.9252642837625905.
[I 2026-03-08 16:30:49,365] Trial 2 finished with value: 0.6558123007622629 and parameters: {'ngram_max': 2, 'alpha': 9.841772188564112e-09}. Best is trial 1 with value: 0.9252642837625905.
[I 2026-03-08 16:30:56,676] Trial 3 finished with value: 0.6558123007622629 and parameters: {'ngram_max': 2, 'alpha': 1.0075492673763716e-09}. Best is trial 1 with value: 0.9252642837625905.
[I 2026-03-08 16:31:04,336] Trial 4 finished with value: 0.8709483658239231 and parameters: {'ngram_max': 3, 'alpha': 5.93583170167752e-08}. Best is trial 1 with value: 0.9252642837625905.
[I 2026-03-08 16:31:15,616] Trial 5 finished with valu

## Final model evaluation

In [41]:
best_params = study.best_params
eval_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb', 
        ngram_range=(1, best_params['ngram_max'])
    )),
    ('classifier', MultinomialNB(alpha=best_params['alpha']))
])

In [42]:
eval_pipeline.fit(X_train_full, y_train_full)

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [43]:
y_pred = eval_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

                                     precision    recall  f1-score   support

                           Алкоголь       0.99      0.98      0.98      2712
                            Бакалія       0.88      0.89      0.89       944
Готова їжа, снеки та напівфабрикати       0.91      0.81      0.86      2659
                  Гігієна та догляд       0.99      0.99      0.99      2561
                        Електроніка       0.99      0.97      0.98       114
                         Канцелярія       0.93      0.85      0.89       471
                              Книги       0.87      0.98      0.92       575
                      М'ясо та риба       0.93      0.96      0.95      1698
                   Молочні продукти       0.96      0.95      0.96      1176
                              Напої       0.93      0.97      0.95      1292
                     Одяг та взуття       1.00      0.99      1.00      1335
                           Солодощі       0.90      0.96      0.93      210

Our results satisfy us, so we'll proceed to final stages.

# Final training

In [44]:
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb', 
        ngram_range=(1, best_params['ngram_max'])
    )),
    ('classifier', MultinomialNB(alpha=best_params['alpha']))
])

In [45]:
final_pipeline.fit(df['preprocessed'], df['category'])

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


# Exporting

In [48]:
joblib.dump(final_pipeline, MODEL_EXPORT_PATH)

['../app/models/exports/model.joblib']

In [52]:
with open(CATEGORIES_EXPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(categories_to_leave, f, ensure_ascii=False, indent=4)